In [1]:
import os
import requests
from groq import Groq
import gradio as gr
from dotenv import load_dotenv

# =====================================
# LOAD ENV
# =====================================

load_dotenv()

groq_client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

WEATHER_KEY = os.getenv("WEATHER_API_KEY")


# =====================================
# TOOL: WEATHER API
# =====================================

def get_weather(city):

    url = f"http://api.weatherapi.com/v1/current.json?key={WEATHER_KEY}&q={city}"

    response = requests.get(url)

    data = response.json()

    result = {

        "city": data["location"]["name"],

        "temperature": data["current"]["temp_c"],

        "condition": data["current"]["condition"]["text"],

        "humidity": data["current"]["humidity"]
    }

    return result


# =====================================
# MCP DECISION USING LLM
# =====================================

def analyze_query(user_input):

    prompt = f"""
You are an AI agent.

Decide:

1. whether weather tool is needed
2. extract city name

Return JSON:

{{
 "use_tool": true/false,
 "city": "city name or null"
}}

Query:
"{user_input}"
"""

    response = groq_client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[

            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        decision = json.loads(output)

    except:

        decision = {
            "use_tool": False,
            "city": None
        }

    return decision


# =====================================
# MCP AGENT
# =====================================

def weather_agent(user_input):

    decision = analyze_query(user_input)

    if decision["use_tool"]:

        weather = get_weather(decision["city"])

        return f"""
Weather Report 🌤

City: {weather['city']}

Temperature: {weather['temperature']}°C

Condition: {weather['condition']}

Humidity: {weather['humidity']}%
"""

    else:

        return "Ask about weather like: weather in Delhi"


# =====================================
# GRADIO UI
# =====================================

demo = gr.Interface(

    fn=weather_agent,

    inputs="text",

    outputs="text",

    title="MCP Weather Agent",

    description="Ask weather using MCP + Groq"

)

demo.launch()

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable